# DFDN Custom Implementation (Model B)

## Data Preperation

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import random
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and prepare data
train_data = pd.read_csv("Data/train_standard.csv")
test_data = pd.read_csv("Data/test_standard.csv")

# Separate features and labels
X_train = train_data.drop(columns=["target"]) 
y_train = train_data["target"]
X_test = test_data.drop(columns=["target"])
y_test = test_data["target"]

# Normalize feature data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Split training data into train and validation sets
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train_tensor, y_train_tensor, test_size=0.2, random_state=42
)

# Create DataLoaders
train_ds = DataLoader(TensorDataset(X_train_split, y_train_split), batch_size=32, shuffle=True)
val_ds = DataLoader(TensorDataset(X_val, y_val), batch_size=32)
test_ds = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32)


## Linear Neural Network embedding layer

In [33]:
# Define the neural feature extractor
class NeuralFeatureExtractor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super(NeuralFeatureExtractor, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return x

## Tree in Decision Forest implemetation

In [34]:
# Define a differentiable decision tree with soft routing and trainable leaves
class DNDFTree(nn.Module):
    def __init__(self, depth, input_dim, num_classes, feature_subset=None):
        super(DNDFTree, self).__init__()
        if feature_subset is None:
            self.feature_subset = torch.arange(input_dim)
        else:
            self.feature_subset = feature_subset
        self.depth = depth
        self.num_classes = num_classes
        self.num_leaf_nodes = 2 ** depth

        # Adjust input_dim based on feature_subset
        self.decision_layer = nn.Linear(len(self.feature_subset), self.num_leaf_nodes)
        self.leaf_distributions = nn.Parameter(torch.rand(self.num_leaf_nodes, num_classes))

    def forward(self, x):
        # Select the feature subset
        x = x[:, self.feature_subset]

        batch_size = x.size(0)
        decision_logits = self.decision_layer(x)
        decision_probs = torch.sigmoid(decision_logits)

        # Compute routing probabilities
        mu = torch.ones(batch_size, 1, device=x.device)
        for d in range(self.depth):
            indices = torch.arange(2 ** d).to(x.device)
            probs = decision_probs[:, indices]
            mu = mu.unsqueeze(2)
            mu = torch.cat([mu * probs.unsqueeze(2), mu * (1 - probs).unsqueeze(2)], dim=2)
            mu = mu.view(batch_size, -1)
        mu = mu.view(batch_size, self.num_leaf_nodes)

        # Final leaf probabilities
        leaf_distributions = F.softmax(self.leaf_distributions, dim=-1)
        output = torch.matmul(mu, leaf_distributions)
        return output


## Model implementation 
### Combining embedding layer with forest of decision trees

In [35]:
# DNDF model with a forest of trees
class DNDF(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_trees=5, tree_depth=3):
        super(DNDF, self).__init__()
        self.num_classes = num_classes
        
        # Neural network feature extractor
        self.feature_extractor = NeuralFeatureExtractor(input_dim, hidden_dim)
        
        # Generate feature subsets for each tree
        feature_indices = np.arange(hidden_dim)
        self.trees = nn.ModuleList([
            DNDFTree(
                tree_depth, 
                len(feature_subset := torch.tensor(
                    np.random.choice(feature_indices, size=int(0.7 * hidden_dim), replace=False)
                )), 
                num_classes, 
                feature_subset=feature_subset
            ) 
            for _ in range(num_trees)
        ])

    def forward(self, x):
        # Pass data through the feature extractor
        x = self.feature_extractor(x)
        
        # Aggregate predictions from each tree in the forest
        tree_outputs = [tree(x) for tree in self.trees]
        forest_output = torch.mean(torch.stack(tree_outputs), dim=0)
        
        return forest_output

# Model parameters
input_dim = X_train.shape[1]
hidden_dim = 64
num_classes = len(y_train.unique())
num_trees = 5
tree_depth = 3

## Model initialisation and training setup

In [36]:
# Initialize the DNDF model
model = DNDF(input_dim, hidden_dim, num_classes, num_trees, tree_depth).to(device)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training function
def train(model, train_loader, val_loader, criterion, optimizer, epochs=10, patience=5):
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Training phase
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                val_output = model(val_x)
                val_loss += criterion(val_output, val_y).item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch + 1}, Training Loss: {loss.item():.4f}, Validation Loss: {val_loss:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered")
                break

## Model Implementation

In [37]:
# Training the model
train(model, train_ds, val_ds, criterion, optimizer, epochs=100)


Epoch 1, Training Loss: 0.6920, Validation Loss: 0.6885
Epoch 2, Training Loss: 0.6571, Validation Loss: 0.6606
Epoch 3, Training Loss: 0.6160, Validation Loss: 0.6347
Epoch 4, Training Loss: 0.5996, Validation Loss: 0.6206
Epoch 5, Training Loss: 0.6075, Validation Loss: 0.6116
Epoch 6, Training Loss: 0.5881, Validation Loss: 0.6109
Epoch 7, Training Loss: 0.5590, Validation Loss: 0.6106
Epoch 8, Training Loss: 0.5785, Validation Loss: 0.6100
Epoch 9, Training Loss: 0.5695, Validation Loss: 0.6093
Epoch 10, Training Loss: 0.6564, Validation Loss: 0.6089
Epoch 11, Training Loss: 0.5582, Validation Loss: 0.6088
Epoch 12, Training Loss: 0.5220, Validation Loss: 0.6088
Epoch 13, Training Loss: 0.5697, Validation Loss: 0.6087
Epoch 14, Training Loss: 0.5672, Validation Loss: 0.6087
Epoch 15, Training Loss: 0.5518, Validation Loss: 0.6086
Epoch 16, Training Loss: 0.5442, Validation Loss: 0.6086
Epoch 17, Training Loss: 0.5907, Validation Loss: 0.6086
Epoch 18, Training Loss: 0.5412, Validat

## Model Evaluation setup

In [39]:
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, accuracy_score, recall_score, f1_score, confusion_matrix

def compute_auprc(y_true, y_probs):
    precision, recall, _ = precision_recall_curve(y_true, y_probs[:, 1], pos_label=1)
    auprc = auc(recall, precision)
    return auprc

def compute_auroc(y_true, y_probs):
    return roc_auc_score(y_true, y_probs[:, 1])

def compute_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def compute_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, average="binary")

def compute_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="binary")

def evaluate(model, test_loader):
    model.eval()
    y_true = []
    y_pred = []
    y_probs = []
    
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            output = model(batch_x)
            y_true.extend(batch_y.tolist())
            
            # Get predicted class (highest probability)
            _, predicted = torch.max(output, 1)
            y_pred.extend(predicted.tolist())
            
            # Get predicted probabilities
            y_probs.extend(torch.softmax(output, dim=1).cpu().tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_probs = np.array(y_probs)

    # Calculate metrics
    auprc = compute_auprc(y_true, y_probs)
    auroc = compute_auroc(y_true, y_probs)
    accuracy = compute_accuracy(y_true, y_pred)
    recall = compute_recall(y_true, y_pred)
    f1 = compute_f1(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    # Print metrics
    print(f"AUPRC: {auprc:.4f}")
    print(f"AUROC: {auroc:.4f}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("Confusion Matrix:\n", cm)

    return auprc, auroc, accuracy, recall, f1, cm


## Model Evaluation

In [40]:
# Evaluate the model
evaluate(model, test_ds)

AUPRC: 0.3475
AUROC: 0.8369
Accuracy: 77.33%
Recall: 0.6744
F1 Score: 0.3412
Confusion Matrix:
 [[353  98]
 [ 14  29]]


(0.3474732426732882,
 0.836899912339504,
 0.7732793522267206,
 0.6744186046511628,
 0.3411764705882353,
 array([[353,  98],
        [ 14,  29]], dtype=int64))